In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# Assuming 'df_all_compounds' is your DataFrame from the previous step
# containing the activities from those 976 assays.

# 1. Extract all unique molecule IDs from those assays
unique_mol_ids = df_all_compounds['molecule_chembl_id'].dropna().unique().tolist()
print(f"Total unique molecules tested across these assays: {len(unique_mol_ids)}")

# 2. Fetch molecule properties in batches to avoid API timeouts
molecule = new_client.molecule
mol_data = []
batch_size = 100

print("Fetching molecule data from ChEMBL... this might take a minute.")
for i in range(0, len(unique_mol_ids), batch_size):
    batch_ids = unique_mol_ids[i:i+batch_size]
    # We only request what we need to save bandwidth
    res = molecule.filter(molecule_chembl_id__in=batch_ids).only(['molecule_chembl_id', 'molecule_properties'])
    mol_data.extend(list(res))
    
df_mols = pd.DataFrame(mol_data)

# 3. Extract the formula (it is nested inside a dictionary in the API response)
def get_formula(props):
    if isinstance(props, dict) and 'full_molformula' in props:
        return props['full_molformula']
    return None

df_mols['formula'] = df_mols['molecule_properties'].apply(get_formula)

# 4. Filter for Platinum ('Pt') in the formula
# Note: Searching 'Pt' is safe in formulas (doesn't clash with other elements like 'P' or 't')
df_pt_mols = df_mols[df_mols['formula'].str.contains('Pt', na=False)].copy()
print(f"Found {len(df_pt_mols)} unique Platinum-containing molecules.")

# 5. Merge the valid Pt-molecules back with your original activity data
df_pt_activities = pd.merge(df_all_compounds, df_pt_mols, on='molecule_chembl_id')

# 6. Apply your inactivity threshold (e.g., > 50,000 nM)
df_pt_activities['standard_value'] = pd.to_numeric(df_pt_activities['standard_value'], errors='coerce')
inactives = df_pt_activities[df_pt_activities['standard_value'] > 50000]

print(f"Found {len(inactives)} strictly inactive Pt-compounds from matching assays!")

In [ ]:
display(inactives[['molecule_chembl_id']].drop_duplicates())

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# Assuming 'df_all_compounds' still holds the activities from your 976 assays 
# from the earlier script.

# 1. Do a global substructure search for ALL compounds in ChEMBL containing a Pt atom
print("Querying ChEMBL for all molecules containing a Platinum atom...")
pt_search = new_client.substructure.filter(smiles='[#78]*').only(['molecule_chembl_id'])

# Extract just the IDs into a list
all_pt_chembl_ids = [mol['molecule_chembl_id'] for mol in pt_search]
print(f"Found {len(all_pt_chembl_ids)} total Pt-containing compounds in the entire ChEMBL database.")

# 2. Cross-reference: Filter your assay results to ONLY include molecules in the master Pt list
df_pt_activities = df_all_compounds[df_all_compounds['molecule_chembl_id'].isin(all_pt_chembl_ids)].copy()
print(f"Found {len(df_pt_activities)} Pt-compound activities tested inside your 976 assays.")

# 3. Apply your inactivity threshold (e.g., > 50,000 nM)
df_pt_activities['standard_value'] = pd.to_numeric(df_pt_activities['standard_value'], errors='coerce')
inactives = df_pt_activities[df_pt_activities['standard_value'] > 50000]

print(f"Success! Found {len(inactives)} strictly inactive Pt-compounds from matching assays.")

# Optional: Save to CSV to inspect them
inactives.to_csv("inactive_pt_compounds.csv", index=False)

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# 1. Fetch ALL molecules with 'Pt' in their formula directly from the API backend
# This avoids the substructure bugs and offloads the search to ChEMBL's servers.
print("Fetching global Pt compounds by formula...")
molecule = new_client.molecule

# Using Django-style __contains filter directly on the API
pt_mols = molecule.filter(
    molecule_properties__full_molformula__contains='Pt'
).only(['molecule_chembl_id', 'molecule_properties'])

pt_chembl_ids = [mol['molecule_chembl_id'] for mol in pt_mols if mol.get('molecule_properties')]
print(f"Found {len(pt_chembl_ids)} Platinum compounds in ChEMBL globally.")

# 2. Fetch the activity data for ALL of these Pt-compounds
print("Fetching activity data for global Pt-compounds...")
activity = new_client.activity

# Batching the activity requests to avoid timeouts
batch_size = 50
pt_activities = []

for i in range(0, len(pt_chembl_ids), batch_size):
    batch_ids = pt_chembl_ids[i:i+batch_size]
    res = activity.filter(
        molecule_chembl_id__in=batch_ids,
        standard_type__in=['IC50', 'GI50'],
        standard_units='nM'
    ).only(['molecule_chembl_id', 'target_chembl_id', 'standard_value', 'assay_chembl_id'])
    pt_activities.extend(list(res))

df_pt_all = pd.DataFrame(pt_activities)

# 3. Filter for strictly Inactive Pt-compounds
df_pt_all['standard_value'] = pd.to_numeric(df_pt_all['standard_value'], errors='coerce')
df_inactives = df_pt_all[df_pt_all['standard_value'] > 50000].copy()

print(f"Found {len(df_inactives)} inactive Pt-compound assay results globally!")

# 4. (Optional but Recommended) Cross-reference targets
# If you have a list of target_chembl_ids (cell lines) where your ACTIVES were tested:
# my_active_targets = ['CHEMBL612545', 'CHEMBL612546'] # Example HeLa / MCF-7 IDs
# final_fair_comparison_inactives = df_inactives[df_inactives['target_chembl_id'].isin(my_active_targets)]

In [ ]:
display(df_inactives[['molecule_chembl_id', 'target_chembl_id', 'assay_chembl_id', 'standard_value']].drop_duplicates())

In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import time

# Cell 2: Fetch all global Platinum compounds
print("1. Fetching global Pt compounds by formula...")
molecule = new_client.molecule

# Query ChEMBL for molecules with Pt in the formula
pt_mols = molecule.filter(
    molecule_properties__full_molformula__contains='Pt'
).only(['molecule_chembl_id'])

pt_chembl_ids = [mol['molecule_chembl_id'] for mol in pt_mols]
print(f"-> Found {len(pt_chembl_ids)} Platinum compounds in ChEMBL globally.\n")

# Cell 3: Fetch Activity Data in Batches
print("2. Fetching activity data (IC50/GI50 in nM) for all Pt-compounds...")
activity = new_client.activity

batch_size = 100
pt_activities = []

# Loop through the IDs in batches to prevent API timeouts
for i in range(0, len(pt_chembl_ids), batch_size):
    batch_ids = pt_chembl_ids[i:i+batch_size]
    
    # We use a try-except block in case the ChEMBL API briefly drops connection
    try:
        res = activity.filter(
            molecule_chembl_id__in=batch_ids,
            standard_type__in=['IC50', 'GI50'],
            standard_units='nM'
        ).only(['molecule_chembl_id', 'target_chembl_id', 'standard_value', 'assay_chembl_id'])
        
        pt_activities.extend(list(res))
    except Exception as e:
        print(f"Warning: Batch {i} to {i+batch_size} failed. Skipping. Error: {e}")
        time.sleep(2) # Brief pause if the server complains

df_raw_activities = pd.DataFrame(pt_activities)
print(f"-> Retrieved {len(df_raw_activities)} total assay results.\n")

# Cell 4: Clean and Aggregate the Data
print("3. Cleaning and aggregating data...")

# Convert standard_value to numeric, forcing errors (like text/symbols) to NaN
df_raw_activities['standard_value'] = pd.to_numeric(df_raw_activities['standard_value'], errors='coerce')

# Drop rows where the value is missing
df_clean = df_raw_activities.dropna(subset=['standard_value'])

# Aggregate: Group by molecule ID and find the MINIMUM (most potent) IC50
# We also count unique assays to see how well-tested each compound is
df_agg = df_clean.groupby('molecule_chembl_id').agg(
    best_ic50_nM=('standard_value', 'min'),
    total_assays=('assay_chembl_id', 'nunique')
).reset_index()

# Cell 5: Apply Classification Logic
print("4. Classifying compounds for Machine Learning...")

def classify_drug(ic50):
    if ic50 < 10000:   # Highly to moderately active in at least one assay
        return 'Active'
    elif ic50 >= 50000: # Failed to show activity in ALL tested assays
        return 'Inactive'
    else:               # Gray area (10,000 - 50,000) - best to exclude from training
        return 'Inconclusive'

df_agg['ml_class'] = df_agg['best_ic50_nM'].apply(classify_drug)

# Separate the dataframes for your model
df_actives = df_agg[df_agg['ml_class'] == 'Active'].copy()
df_inactives = df_agg[df_agg['ml_class'] == 'Inactive'].copy()
df_inconclusive = df_agg[df_agg['ml_class'] == 'Inconclusive'].copy()

# Cell 6: Final Summary & Export
print("\n=== FINAL DATASET SUMMARY ===")
print(f"Total Unique Pt-Compounds Tested: {len(df_agg)}")
print(f"✅ Global Actives (Min IC50 < 10µM): {len(df_actives)}")
print(f"❌ Global Inactives (Min IC50 > 50µM): {len(df_inactives)}")
print(f"⚠️ Inconclusive (10µM - 50µM): {len(df_inconclusive)}")

# Optional: Check Cisplatin specifically to ensure it landed in the right bucket
cisplatin_id = 'CHEMBL11359'
if cisplatin_id in df_agg['molecule_chembl_id'].values:
    cis_data = df_agg[df_agg['molecule_chembl_id'] == cisplatin_id].iloc[0]
    print(f"\nSanity Check -> Cisplatin ({cisplatin_id}): Best IC50 = {cis_data['best_ic50_nM']} nM | Class = {cis_data['ml_class']}")

# Save to CSV for your descriptor calculations
df_actives.to_csv("pt_actives_dataset.csv", index=False)
df_inactives.to_csv("pt_inactives_dataset.csv", index=False)
print("\nSaved 'pt_actives_dataset.csv' and 'pt_inactives_dataset.csv' to your working directory.")

In [ ]:
display(df_inactives)

In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import time

# Cell 2: Fetch global Platinum compounds AND their descriptive info
print("1. Fetching global Pt compounds (IDs, Names, and Formulas)...")
molecule = new_client.molecule

# Query ChEMBL for molecules with Pt in the formula, asking for extra descriptive fields
pt_mols = molecule.filter(
    molecule_properties__full_molformula__contains='Pt'
).only(['molecule_chembl_id', 'pref_name', 'molecule_properties'])

mol_info_list = []
pt_chembl_ids = []

for mol in pt_mols:
    mol_id = mol.get('molecule_chembl_id')
    pref_name = mol.get('pref_name') # Will be None if it's an unnamed experimental compound
    
    # Safely extract the formula
    props = mol.get('molecule_properties')
    formula = props.get('full_molformula') if props else None
    
    pt_chembl_ids.append(mol_id)
    mol_info_list.append({
        'molecule_chembl_id': mol_id,
        'pref_name': pref_name,
        'formula': formula
    })

df_mol_info = pd.DataFrame(mol_info_list)
print(f"-> Found {len(pt_chembl_ids)} Platinum compounds in ChEMBL globally.\n")

# Cell 3: Fetch Activity Data in Batches
print("2. Fetching activity data (IC50/GI50 in nM) for all Pt-compounds...")
activity = new_client.activity

batch_size = 100
pt_activities = []

# Loop through the IDs in batches to prevent API timeouts
for i in range(0, len(pt_chembl_ids), batch_size):
    batch_ids = pt_chembl_ids[i:i+batch_size]
    
    try:
        res = activity.filter(
            molecule_chembl_id__in=batch_ids,
            standard_type__in=['IC50', 'GI50'],
            standard_units='nM'
        ).only(['molecule_chembl_id', 'target_chembl_id', 'standard_value', 'assay_chembl_id'])
        
        pt_activities.extend(list(res))
    except Exception as e:
        print(f"Warning: Batch {i} to {i+batch_size} failed. Skipping. Error: {e}")
        time.sleep(2) 

df_raw_activities = pd.DataFrame(pt_activities)
print(f"-> Retrieved {len(df_raw_activities)} total assay results.\n")

# Cell 4: Clean, Aggregate, and Merge Descriptive Data
print("3. Cleaning, aggregating, and adding descriptions...")

# Convert standard_value to numeric
df_raw_activities['standard_value'] = pd.to_numeric(df_raw_activities['standard_value'], errors='coerce')
df_clean = df_raw_activities.dropna(subset=['standard_value'])

# Aggregate: Group by molecule ID
df_agg = df_clean.groupby('molecule_chembl_id').agg(
    best_ic50_nM=('standard_value', 'min'),
    total_assays=('assay_chembl_id', 'nunique')
).reset_index()

# MERGE: Attach the names and formulas to the aggregated data!
df_agg = pd.merge(df_agg, df_mol_info, on='molecule_chembl_id', how='left')

# Reorder columns so the names/formulas are right next to the ID for easy reading
df_agg = df_agg[['molecule_chembl_id', 'pref_name', 'formula', 'best_ic50_nM', 'total_assays']]


# Cell 5: Apply Classification Logic
print("4. Classifying compounds for Machine Learning...")

def classify_drug(ic50):
    if ic50 < 10000:   # Active
        return 'Active'
    elif ic50 >= 50000: # Inactive
        return 'Inactive'
    else:               # Inconclusive (Gray area)
        return 'Inconclusive'

df_agg['ml_class'] = df_agg['best_ic50_nM'].apply(classify_drug)

df_actives = df_agg[df_agg['ml_class'] == 'Active'].copy()
df_inactives = df_agg[df_agg['ml_class'] == 'Inactive'].copy()
df_inconclusive = df_agg[df_agg['ml_class'] == 'Inconclusive'].copy()

# Cell 6: Final Summary & Export
print("\n=== FINAL DATASET SUMMARY ===")
print(f"Total Unique Pt-Compounds Tested: {len(df_agg)}")
print(f"✅ Global Actives (Min IC50 < 10µM): {len(df_actives)}")
print(f"❌ Global Inactives (Min IC50 > 50µM): {len(df_inactives)}")
print(f"⚠️ Inconclusive (10µM - 50µM): {len(df_inconclusive)}")

# Sanity check for Cisplatin using the new pref_name column
cis_check = df_agg[df_agg['pref_name'].str.contains('CISPLATIN', case=False, na=False)]
if not cis_check.empty:
    cis_data = cis_check.iloc[0]
    print(f"\nSanity Check -> {cis_data['pref_name']} ({cis_data['molecule_chembl_id']}): Formula = {cis_data['formula']} | Best IC50 = {cis_data['best_ic50_nM']} nM | Class = {cis_data['ml_class']}")

df_actives.to_csv("pt_actives_dataset.csv", index=False)
df_inactives.to_csv("pt_inactives_dataset.csv", index=False)
print("\nSaved datasets with Descriptions/Formulas to your working directory.")

In [ ]:
display(df_inactives)

In [4]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
import time

molecule = new_client.molecule
activity = new_client.activity

print("1. Gathering unique Platinum ChEMBL IDs...")
pt_ids = set()

# Molecular Formula Search
formula_results = molecule.filter(molecule_properties__full_molformula__contains='Pt').only(['molecule_chembl_id'])
pt_ids.update([comp['molecule_chembl_id'] for comp in formula_results if comp.get('molecule_chembl_id')])

# Text Search
for keyword in ['platinum', 'platin']:
    text_results = molecule.search(keyword)
    pt_ids.update([comp['molecule_chembl_id'] for comp in text_results if comp.get('molecule_chembl_id')])

pt_chembl_ids = list(pt_ids)
print(f"Found {len(pt_chembl_ids)} unique Platinum-containing compounds.")

print("\n2. Fetching Molecule Metadata (Formula, Name)...")
mol_data = []
batch_size = 50

# Fetch molecule metadata in chunks
for i in range(0, len(pt_chembl_ids), batch_size):
    batch = pt_chembl_ids[i:i+batch_size]
    res = molecule.filter(molecule_chembl_id__in=batch).only(['molecule_chembl_id', 'pref_name', 'molecule_properties'])
    
    for m in res:
        props = m.get('molecule_properties')
        formula = props.get('full_molformula', 'No Formula') if props else 'No Formula'
        
        mol_data.append({
            'molecule_chembl_id': m['molecule_chembl_id'],
            'molecule_name': m.get('pref_name', 'No Name'),
            'formula': formula
        })
    time.sleep(0.2)

mol_df = pd.DataFrame(mol_data)

print("\n3. Fetching Activity Data (All Raw IC50 Units)...")
act_data = []
for i in range(0, len(pt_chembl_ids), batch_size):
    batch = pt_chembl_ids[i:i+batch_size]
    res = activity.filter(
        molecule_chembl_id__in=batch,
        assay_type__in=['F', 'T'],
        standard_type__iexact='IC50'
    ).only([
        'molecule_chembl_id', 'standard_relation', 'standard_value', 
        'standard_units', 'assay_description', 'target_pref_name', 'pchembl_value'
    ])
    act_data.extend(res)
    time.sleep(0.2)

act_df = pd.DataFrame(act_data)
print(f"Fetched {len(act_df)} raw assay records.")

print("\n4. Merging and Formatting Output DataFrame...")
# Merge the two DataFrames based on the ChEMBL ID
final_df = pd.merge(act_df, mol_df, on='molecule_chembl_id', how='left')

# Reorder columns for optimal readability
columns_order = [
    'molecule_chembl_id', 
    'molecule_name', 
    'formula', 
    'target_pref_name',
    'assay_description', 
    'standard_relation', 
    'standard_value', 
    'standard_units',
    'pchembl_value'
]

# Keep only the columns that successfully pulled from the API
final_df = final_df[[col for col in columns_order if col in final_df.columns]]

# Sort by ChEMBL ID so you can see all assays for a single compound grouped together
final_df = final_df.sort_values(by='molecule_chembl_id')

print(f"\nFinal dataset contains {len(final_df)} records.")

# Save to CSV for manual inspection/custom processing
csv_filename = "platinum_raw_all_units.csv"
final_df.to_csv(csv_filename, index=False)
print(f"Data successfully saved to {csv_filename}!")

# Display a breakdown of the units you collected so you know what you are working with
print("\nSummary of units found in your dataset:")
print(final_df['standard_units'].value_counts(dropna=False))
print("-" * 40)

# Display the first few rows in Jupyter
display(final_df.head(10))

1. Gathering unique Platinum ChEMBL IDs...
Found 1127 unique Platinum-containing compounds.

2. Fetching Molecule Metadata (Formula, Name)...

3. Fetching Activity Data (All Raw IC50 Units)...
Fetched 2883 raw assay records.

4. Merging and Formatting Output DataFrame...

Final dataset contains 2883 records.
Data successfully saved to platinum_raw_all_units.csv!

Summary of units found in your dataset:
standard_units
nM         2648
ug.mL-1     235
Name: count, dtype: int64
----------------------------------------


,molecule_chembl_id,molecule_name,formula,target_pref_name,assay_description,standard_relation,standard_value,standard_units,pchembl_value
1042,CHEMBL100222,"1,10-Phenanthrolineplatinum (II) Benzoylthiour...",C24H23Cl2N4O3PtS+,Plasmodium falciparum,In vitro antimalarial activity against chloroq...,None,666.0,nM,None
1041,CHEMBL100222,"1,10-Phenanthrolineplatinum (II) Benzoylthiour...",C24H23Cl2N4O3PtS+,Plasmodium falciparum,In vitro antimalarial activity against chloroq...,=,594.0,nM,6.23
21,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,SK-OV-3,In vitro antitumor activity against SK-OV-3 ce...,=,72000.0,nM,4.14
23,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,41M,In vitro antitumor activity against 41McisR ce...,=,33000.0,nM,4.48
22,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,41M,In vitro antitumor activity against 41M cell l...,=,5400.0,nM,5.27
26,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,A2780,In vitro antitumor activity against A2780 cell...,=,6200.0,nM,5.21
20,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,HX62 cell line,In vivo antitumor activity against HX/62 cell ...,=,110000.0,nM,None
27,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,A2780,In vitro antitumor activity against A2780cisR ...,=,64000.0,nM,4.19
24,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,CH1,In vitro antitumor activity against CH1 cell l...,=,12500.0,nM,4.90
25,CHEMBL102282,Platinum complex,CH8Cl2N2O2Pt,CH1,In vitro antitumor activity against CH1cisR ce...,=,30500.0,nM,4.52


In [10]:
import numpy as np

print("1. Converting units to Micromolar (µM)...")

def standardize_to_uM(row):
    val = row['standard_value']
    unit = row['standard_units']
    
    # 1. Handle missing data
    if pd.isna(val) or pd.isna(unit):
        return np.nan
        
    # 2. Force the value to be a float so we can do math on it!
    try:
        val = float(val)
    except ValueError:
        # If the value is weird text that can't be a number, discard it
        return np.nan
        
    unit = str(unit).strip()
    
    # 3. Molar conversions
    if unit == 'nM':
        return val / 1000.0
    elif unit in ['uM', 'umol/L', 'µM']:
        return val
    elif unit == 'mM':
        return val * 1000.0
    elif unit == 'M':
        return val * 1000000.0
    else:
        # Returns NaN for mass-based units
        return np.nan

# Now re-run the apply function
final_df['IC50_uM'] = final_df.apply(standardize_to_uM, axis=1)

# Drop any rows where the conversion failed (e.g., incompatible units)
working_df = final_df.dropna(subset=['IC50_uM']).copy()
print(f"Retained {len(working_df)} assays with valid molar units.")

print("\n2. Applying the 100 µM threshold...")
# Create boolean columns representing your conditions
# True evaluates to 1 and False evaluates to 0, which makes counting easy
working_df['is_active'] = working_df['IC50_uM'] < 50
working_df['is_inactive'] = working_df['IC50_uM'] >= 50

print("3. Grouping and summarizing by Compound ID...")
# Group by the identifiers and sum the boolean flags to get the total counts
summary_df = working_df.groupby(['molecule_chembl_id', 'molecule_name', 'formula']).agg(
    total_assays=('IC50_uM', 'count'),
    median_IC50_uM=('IC50_uM', 'median'), # Helpful context metric to keep
    Active=('is_active', 'sum'),
    Inactive=('is_inactive', 'sum')
).reset_index()

# Sort by the most heavily tested compounds first
summary_df = summary_df.sort_values(by='total_assays', ascending=False)

print(f"\nSuccessfully summarized down to {len(summary_df)} unique Platinum compounds.")

# Save the summarized consensus dataset
summary_csv = "platinum_consensus_summary.csv"
#summary_df.to_csv(summary_csv, index=False)
print(f"Summary saved to {summary_csv}")

# Display the top results
display(summary_df.head(10))

1. Converting units to Micromolar (µM)...
Retained 2632 assays with valid molar units.

2. Applying the 100 µM threshold...
3. Grouping and summarizing by Compound ID...

Successfully summarized down to 326 unique Platinum compounds.
Summary saved to platinum_consensus_summary.csv


,molecule_chembl_id,molecule_name,formula,total_assays,median_IC50_uM,Active,Inactive
20,CHEMBL11359,CISPLATIN,H6Cl2N2Pt+2,870,16.174215,613,257
29,CHEMBL1351,CARBOPLATIN,C6H12N2O4Pt,69,20.000000,52,17
35,CHEMBL1386,TRANSPLATIN,H6Cl2N2Pt,28,77.000000,7,21
19,CHEMBL111089,trans-[{PtCl(NH3)2}-mu-{NH2(CH2)6NH2}-cis-{PtC...,C6H20Cl3N5Pt2,16,0.975000,16,0
232,CHEMBL321094,Platinum complex,C5H9Cl2N2O2Pt+,16,5.100000,16,0
18,CHEMBL108133,trans-[{PtCl(NH3)2}-mu-{NH2(CH2)6NH2}-trans-{P...,C6H22Cl2N6Pt2+2,16,1.850000,15,1
12,CHEMBL103570,"trans-a-ammine-b,d-dichloro-f-(cyclohexylamine...",C6H16Cl2N2O2Pt,16,2.050000,16,0
231,CHEMBL321068,"trans-a-ammine-b,d-dichloro-f-(2-propylamine)-...",C3H12Cl2N2O2Pt,16,4.800000,15,1
15,CHEMBL104893,AMMONIO(DICHLORO)PLATINUM(CYCLOHEXYL)AMMONIUM,C6H14Cl2N2Pt,16,1.350000,16,0
8,CHEMBL103345,"trans-a-ammine-b,d-dichloro-f-(2-methyl1-2-pro...",C4H14Cl2N2O2Pt,16,2.600000,15,1


In [12]:
print("Filtering for predominantly inactive compounds...")

# Filter the summary dataframe where the Inactive count is strictly greater than the Active count
inactive_consensus_df = summary_df[summary_df['Inactive'] > summary_df['Active']].copy()

print(f"Found {len(inactive_consensus_df)} Platinum compounds that are mostly inactive.")

# Optional: Sort them by the highest number of inactive assays so the most reliable ones are at the top
inactive_consensus_df = inactive_consensus_df.sort_values(by=['Inactive', 'total_assays'], ascending=[False, False])

# Display the top results
display(inactive_consensus_df.head(30))

# Save this specific inactive list to a new CSV for your ML negative class
inactive_csv = "platinum_predominantly_inactive.csv"
inactive_consensus_df.to_csv(inactive_csv, index=False)
print(f"Inactive consensus data saved to {inactive_csv}")

Filtering for predominantly inactive compounds...
Found 51 Platinum compounds that are mostly inactive.


,molecule_chembl_id,molecule_name,formula,total_assays,median_IC50_uM,Active,Inactive
35,CHEMBL1386,TRANSPLATIN,H6Cl2N2Pt,28,77.000,7,21
310,CHEMBL73466,"2,6-bis(2,2-dimethylpropoxy)-1,3,5,7-tetrathia...",C12H24O2PtS4,12,79.000,2,10
308,CHEMBL72816,"2,6-bis(2,2-dimethyl-3-phenylpropoxy)-1,3,5,7-...",C24H32O2PtS4,12,50.000,2,10
304,CHEMBL72317,"2,6-bis(cyclopropylmethoxy)-1,3,5,7-tetrathia-...",C10H16O2PtS4,12,78.500,3,9
217,CHEMBL312519,"2,6-bis(nonyloxy)-1,3,5,7-tetrathia-4-platinas...",C20H40O2PtS4,12,76.000,3,9
313,CHEMBL75221,"2,6-bis(octyloxy)-1,3,5,7-tetrathia-4-platinas...",C18H36O2PtS4,12,69.500,4,8
14,CHEMBL103847,Trans-Amminedichloro(diethylamine)platinium (II),C4H12Cl2N2Pt,8,50.000,0,8
314,CHEMBL75222,"2,6-bis(hexyloxy)-1,3,5,7-tetrathia-4-platinas...",C14H28O2PtS4,12,67.000,5,7
220,CHEMBL317255,trans-Amminedichloro(quinuclidine)platinium (II),C7H16Cl2N2Pt,8,50.000,1,7
67,CHEMBL1743584,"Cis-[Pt(1,3-diaminepropane)2(O-Succinate)2]",C14H24N4O8Pt,6,200000.000,0,6


Inactive consensus data saved to platinum_predominantly_inactive.csv
